# 03 – Dentro un LLM: token e parola successiva

Usiamo un modello piccolo (**Qwen2.5 0.5B**, mezzo miliardo di parametri) che gira gratis in Colab.
Seguiamo un solo esempio: i giorni della settimana, *"lunedì, martedì, mercoledì,"*.

In questo notebook:
1. vediamo come il testo diventa token;
2. guardiamo le probabilità della parola successiva;
3. generiamo testo un token alla volta;
4. provochiamo un'allucinazione.

Prima di iniziare: **Runtime → Cambia tipo di runtime → T4 GPU**. Funziona anche senza GPU, ma più lentamente.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'   # evita il download che in Colab a volte si blocca
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

nome = 'Qwen/Qwen2.5-0.5B-Instruct'
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(nome)
modello = AutoModelForCausalLM.from_pretrained(nome, torch_dtype='auto').to(dispositivo)

frase = 'lunedì, martedì, mercoledì,'
print('Pronto su', dispositivo)

## 1. Token

Il modello non legge lettere né parole: legge **token**, pezzi di testo con un numero associato.

In [ ]:
def mostra_token(testo):
    ids = tokenizer.encode(testo)
    print(f'{len(ids)} token per: {testo}')
    for i in ids:
        print(f'  {i:>6}  →  [{tokenizer.decode([i])}]')

mostra_token(frase)
mostra_token('Monday, Tuesday, Wednesday,')

L'italiano di solito usa più token dell'inglese: i modelli hanno letto soprattutto testi in inglese.
Più token = più costo e meno spazio nella finestra di contesto.

## 2. La parola successiva

Diamo al modello la frase e guardiamo le probabilità che assegna al token successivo.

In [ ]:
def prossimo_token(testo, quanti=5):
    ids = tokenizer(testo, return_tensors='pt').input_ids.to(dispositivo)
    with torch.no_grad():
        logit = modello(ids).logits[0, -1]
    prob = torch.softmax(logit.float(), dim=-1)
    top = torch.topk(prob, quanti)
    print(testo, '…')
    for p, i in zip(top.values, top.indices):
        pezzo = '[' + tokenizer.decode([i.item()]) + ']'
        print(f'  {pezzo:<16} {p.item():6.1%}  ' + '█' * round(p.item() * 40))

prossimo_token(frase)

Il modello non “sa” che giorno viene dopo: assegna una probabilità a ogni token possibile (oltre 150.000)
e "giovedì" è il più probabile perché l'ha visto migliaia di volte dopo "mercoledì".

## 3. Generare = ripetere

Per scrivere, il modello sceglie un token, lo aggiunge al testo e ricomincia.
Qui lo facciamo a mano, sempre con il token più probabile.

In [ ]:
ids = tokenizer(frase, return_tensors='pt').input_ids.to(dispositivo)

for passo in range(1, 11):
    with torch.no_grad():
        logit = modello(ids).logits[0, -1]
    ids = torch.cat([ids, logit.argmax().view(1, 1)], dim=1)
    print(f'passo {passo:>2}:  {tokenizer.decode(ids[0])}')

## 4. Allucinazioni

Ora usiamo il modello come assistente e gli facciamo due domande sui giorni: una vera e una su un giorno **che non esiste**.

In [ ]:
def chiedi(domanda, token=80):
    messaggi = [{'role': 'user', 'content': domanda}]
    testo = tokenizer.apply_chat_template(messaggi, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(testo, return_tensors='pt').input_ids.to(dispositivo)
    out = modello.generate(ids, max_new_tokens=token, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print(chiedi('Che giorno della settimana viene dopo il mercoledì?'))
print()
print(chiedi('Che giorno della settimana viene dopo il tarvedì?'))

Il "tarvedì" non esiste, ma la risposta può suonare convincente: il modello genera ciò che è **probabile**, non ciò che è **vero**.
Se risponde che non esiste, prova con un altro nome inventato.

I modelli grandi sbagliano meno di questo, ma il principio è lo stesso: **le risposte importanti vanno verificate.**

## Da ricordare

- Testo → token → numeri.
- Un LLM calcola la probabilità del token successivo, e ripete.
- Le allucinazioni sono una conseguenza del meccanismo, non un guasto occasionale.